# Class 02 · Part 2: Using station data

In part 1, `class02_get_station_data.ipynb`, you found the daily data of the station **Hamburg-Fuhlsbüttel**
(since 1936) on the DWD server and downloaded it with `download_dwd_daily`. Now we use it.

In this notebook you

1. get the file of Hamburg with `download_dwd_daily`,
2. read it with **pandas**, the Python package for tables,
3. plot the **time series** of temperature: daily, monthly and yearly,
4. save the plot for your page, and
5. do the same for the **precipitation in Berlin** (exercise).

Remember: run the cells from top to bottom with `Shift + Enter`.

## 1. Packages

`pandas` (short `pd`) works with tables: rows and columns, like a spreadsheet. `download_dwd_daily` is the
function of the course you met in part 1.

In [ ]:
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

from utils import download_dwd_daily

## 2. The file of the station

Hamburg-Fuhlsbüttel is station **1975**. `download_dwd_daily` gives back its file in the folder `data`; it
downloads it only if it is not there yet (e.g. if you start with this notebook).

In [ ]:
hamburg_file = download_dwd_daily(1975)
hamburg_file

The file is a **zip** file: a box with several files inside. Let's look at their names:

In [ ]:
zip_file = zipfile.ZipFile(hamburg_file)
zip_file.namelist()

Most files are descriptions of the station and its instruments (*Metadaten*). The measurements are in the
file that starts with `produkt_klima_tag`. Let's find its name:

In [ ]:
# a loop through the names, keeping the one that starts with "produkt_klima_tag"
for name in zip_file.namelist():
    if name.startswith("produkt_klima_tag"):
        data_name = name

data_name

## 3. Read the data with pandas

The data file is a text file. Its first lines look like this:

```
STATIONS_ID;MESS_DATUM;QN_3;  FX;  FM;QN_4; RSK;RSKF; SDK;SHK_TAG;  NM; VPM;  PM; TMK; UPM; TXK; TNK; TGK;eor
       1975;19360101;-999;-999;-999;    5;   0.1;   1;-999;   0;   7.7;   8.5;  998.00;    7.3;   82.00;    8.8;    5.7;-999;eor
```

So we tell `pd.read_csv` three things:

- `sep=";"`: the values are separated by `;`,
- `skipinitialspace=True`: ignore the spaces before the values,
- `na_values=[-999]`: `-999` means *no measurement*; pandas writes `NaN` (*not a number*) instead.

The result is a **DataFrame**: a table. `.head()` shows its first 5 rows.

In [ ]:
hamburg = pd.read_csv(
    zip_file.open(data_name), sep=";", skipinitialspace=True, na_values=[-999]
)
hamburg.head()

What do the columns mean? The most useful ones (all in the description file *Metadaten_Parameter* of the zip):

| Column | Meaning | Unit |
|---|---|---|
| `MESS_DATUM` | date, as `yyyymmdd` | |
| `TMK` | daily mean temperature (2 m) | °C |
| `TXK`, `TNK` | daily maximum and minimum temperature (2 m) | °C |
| `RSK` | daily precipitation | mm |
| `SDK` | sunshine duration | hours |
| `FM` | daily mean wind speed | m/s |

You get one column with `hamburg["TMK"]`. A column has useful functions, like `.mean()`, `.min()`, `.max()`:

In [ ]:
print("mean temperature:", hamburg["TMK"].mean())
print("coldest day:", hamburg["TMK"].min())
print("warmest day:", hamburg["TMK"].max())

## 4. Dates

For pandas, `19360101` is just a number. `pd.to_datetime` turns it into a real date (first as text, then
with the pattern `%Y%m%d`: year, month, day). We use the dates as the **index** of the table: the labels of
the rows. Then we can select time periods and compute monthly and yearly values.

In [ ]:
hamburg["date"] = pd.to_datetime(hamburg["MESS_DATUM"].astype(str), format="%Y%m%d")
hamburg = hamburg.set_index("date")
hamburg[["TMK", "RSK"]].head()

With dates as index, `.loc[...]` selects a period, e.g. one year:

In [ ]:
hamburg.loc["2025", "TMK"]

Are there days without measurement? `.isna()` is `True` where a value is missing, and `.sum()` counts them:

In [ ]:
hamburg["TMK"].isna().sum()

## 5. Time series of temperature

A **time series** is a plot of a value against time. Daily values of 90 years are many points: the
seasons go up and down every year.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
hamburg["TMK"].plot(ax=ax, linewidth=0.3)
ax.set_ylabel("temperature / °C")
ax.set_title("Hamburg-Fuhlsbüttel, daily mean temperature")
plt.show()

**Monthly and yearly means.** `.resample("MS")` groups the days by month (MS: *month start*),
`.resample("YS")` by year; `.mean()` then averages each group.

In [ ]:
monthly = hamburg["TMK"].resample("MS").mean()
yearly = hamburg["TMK"].resample("YS").mean()

yearly.tail()

Now the plot for your page: monthly and yearly means in one figure. Then we save it next to your page
(`pages/02_station_data/`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, linewidth=0.5, color="lightgray", label="monthly mean")
yearly.plot(ax=ax, linewidth=2, color="firebrick", label="yearly mean")
ax.set_ylabel("temperature / °C")
ax.set_xlabel("year")
ax.set_title("Hamburg-Fuhlsbüttel (DWD station 1975)")
ax.legend()

fig.savefig(
    "/app/pages/02_station_data/hamburg_temperature.png", dpi=100, bbox_inches="tight"
)
plt.show()

**Try it:** which was the warmest year, and which the coldest? `yearly.idxmax()` gives the date of the
largest value, `yearly.idxmin()` of the smallest.

In [ ]:
print("warmest year:", yearly.idxmax().year, round(yearly.max(), 1), "°C")
print("coldest year:", yearly.idxmin().year, round(yearly.min(), 1), "°C")

## 6. Exercise: precipitation in Berlin

Do the same for the **precipitation** (`RSK`, in mm) of the station **Berlin-Tempelhof** (number **433**):

1. Download and read the data, and turn the dates into the index, like in sections 2 to 4.
2. How many days have no precipitation measurement?
3. Precipitation adds up: compute **monthly and yearly sums** with `.resample(...).sum()` instead of `.mean()`.
4. Plot monthly and yearly sums in one figure and save it as
   `/app/pages/02_station_data/berlin_precipitation.png`.
5. Which was the wettest year, and which the driest?

Copy the cells above and change them; add new cells with the `+` button.

In [ ]:
# 1. download and read the data of Berlin-Tempelhof

In [ ]:
# 2. days without precipitation measurement

In [ ]:
# 3. monthly and yearly sums

In [ ]:
# 4. plot and save

In [ ]:
# 5. wettest and driest year

## 7. Your page

Open `pages/02_station_data/my_page.rst` in JupyterLab, write your name, and describe your two plots in a
few sentences. Look at it on http://localhost:8000 → *Classes → 02*.